# Khmer Tokenizer Comparison

Paste one Khmer legal article into the text block, then run the comparison.

In [5]:
%pip install "khmercut[nn]" khmer-nltk datasets pandas pyarrow

  Using cached filelock-4.2.4-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached huggingface_hub-2.3.0-py3-none-any.whl.metadata (16 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached httpx2-2.16.0-py3-none-any.whl.metadata (10 kB)
  Using cached anyio-4.15.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached httpcore2-2.16.0-py3-none-any.whl.metadata (27 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached truststore-0.10.5-py3-none-any.whl.metadata (4.5 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached aiohappyeyeballs-2.7.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 8.3 MB/s  0:00:00
Using cached fsspec-2026.7.0-py3-none-any.whl (206 kB)
   ━━━━━━━━━━━━━━━━━━━

In [6]:
from khmercut import tokenize as cut_crf
from khmercut.nn import tokenize as cut_nn
from khmernltk import word_tokenize as nltk_cut
import time


def flatten_tokens(value):
    if isinstance(value, str):
        return [value]

    tokens = []
    for item in value:
        tokens.extend(flatten_tokens(item))
    return tokens


def clean_tokens(value):
    return [token for token in flatten_tokens(value) if token and not token.isspace()]


tokenizers = {
    "khmercut CRF": lambda text: cut_crf(text),
    "khmercut Neural": lambda text: cut_nn(text, deep=True),
    "khmer-nltk": lambda text: nltk_cut(text, return_tokens=True),
}

## Paste one article here

In [7]:
article_text = """
គណបក្សនយោបាយដែលបានចុះបញ្ជីស្របតាមច្បាប់ស្តីពីគណបក្សនយោបាយ ហើយមានបំណងចូលរួមប្រកួតប្រជែងនៅក្នុងការបោះឆ្នោតជ្រើសរើសក្រុមប្រឹក្សា ត្រូវចាប់ផ្តើមចុះបញ្ជីគណបក្សនយោបាយឈរឈ្មោះបោះឆ្នោត តាមនីតិវិធីដែលរៀបចំដោយគណៈកម្មាធិការជាតិរៀបចំការបោះឆ្នោត យ៉ាងតិចប៉ែតសិប(៨០)ថ្ងៃមុនកាលបរិច្ឆេទនៃការបោះឆ្នោត ។
""".strip()

if not article_text or article_text.startswith("បិទភ្ជាប់"):
    raise ValueError("Paste one real Khmer legal article into article_text first.")

print(f"Characters: {len(article_text):,}")
print(article_text)

Characters: 283
គណបក្សនយោបាយដែលបានចុះបញ្ជីស្របតាមច្បាប់ស្តីពីគណបក្សនយោបាយ ហើយមានបំណងចូលរួមប្រកួតប្រជែងនៅក្នុងការបោះឆ្នោតជ្រើសរើសក្រុមប្រឹក្សា ត្រូវចាប់ផ្តើមចុះបញ្ជីគណបក្សនយោបាយឈរឈ្មោះបោះឆ្នោត តាមនីតិវិធីដែលរៀបចំដោយគណៈកម្មាធិការជាតិរៀបចំការបោះឆ្នោត យ៉ាងតិចប៉ែតសិប(៨០)ថ្ងៃមុនកាលបរិច្ឆេទនៃការបោះឆ្នោត ។


In [8]:
for name, tokenizer in tokenizers.items():
    try:
        # Warm-up
        tokenizer(article_text)

        start = time.perf_counter()
        tokens = clean_tokens(tokenizer(article_text))
        elapsed = (time.perf_counter() - start) * 1000

        print(f"\n{name}")
        print(f"Token count: {len(tokens)}")
        print(f"Time: {elapsed:.2f} ms")
        print(tokens)

    except Exception as e:
        print(f"\n{name}: {type(e).__name__}: {e}")


khmercut CRF
Token count: 48
Time: 3.18 ms
['គណបក្ស', 'នយោបាយ', 'ដែល', 'បាន', 'ចុះបញ្ជី', 'ស្រប', 'តាម', 'ច្បាប់', 'ស្តីពី', 'គណបក្ស', 'នយោបាយ', 'ហើយ', 'មាន', 'បំណង', 'ចូលរួម', 'ប្រកួតប្រជែង', 'នៅក្នុង', 'ការបោះឆ្នោត', 'ជ្រើសរើស', 'ក្រុមប្រឹក្សា', 'ត្រូវ', 'ចាប់ផ្តើម', 'ចុះបញ្ជី', 'គណបក្ស', 'នយោបាយ', 'ឈរឈ្មោះ', 'បោះឆ្នោត', 'តាម', 'នីតិវិធី', 'ដែល', 'រៀបចំ', 'ដោយ', 'គណៈកម្មាធិការ', 'ជាតិ', 'រៀបចំ', 'ការបោះឆ្នោត', 'យ៉ាងតិច', 'ប៉ែត', 'សិប', '(', '៨០', ')', 'ថ្ងៃ', 'មុន', 'កាលបរិច្ឆេទ', 'នៃ', 'ការបោះឆ្នោត', '។']

khmercut Neural
Token count: 67
Time: 5.33 ms
['គណបក្ស', 'នយោបាយ', 'ដែល', 'បាន', 'ចុះ', 'បញ្ជី', 'ស្រប', 'តាម', 'ច្បាប់', 'ស្តី', 'ពី', 'គណបក្ស', 'នយោបាយ', 'ហើយ', 'មាន', 'បំណង', 'ចូល', 'រួម', 'ប្រកួត', 'ប្រជែង', 'នៅ', 'ក្នុង', 'ការ', 'បោះ', 'ឆ្នោត', 'ជ្រើស', 'រើស', 'ក្រុម', 'ប្រឹក្សា', 'ត្រូវ', 'ចាប់ផ្តើម', 'ចុះ', 'បញ្ជី', 'គណបក្ស', 'នយោបាយ', 'ឈរ', 'ឈ្មោះ', 'បោះ', 'ឆ្នោត', 'តាម', 'នីតិ', 'វិធី', 'ដែល', 'រៀបចំ', 'ដោយ', 'គណៈ', 'កម្មាធិការ', 'ជាតិ', 'រៀបចំ', 'ការ', 'បោះ', 'ឆ្នោត', '

In [11]:
from pathlib import Path

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'react-client').is_dir() and (path / 'data-pipeline').is_dir()
)
DATA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data'
TESTABLE_DIR = PROJECT_ROOT / 'data-pipeline' / 'testable'

from datasets import load_dataset

ds = load_dataset("khopilot/khmer-lexicon")

# Save locally
ds.save_to_disk(DATA_DIR / 'khmer-lexicon')

Saving the dataset (1/1 shards): 100%|██████████| 1266/1266 [00:00<00:00, 224942.34 examples/s]
